## Lorenz system

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
%config InlineBackend.figure_format="jpeg"

## Functions and classes

In [ ]:
def lorenz_rhs(u, *, sigma, rho, beta):
    x, y, z = u
    x_dot = sigma * (y - x)
    y_dot = x * (rho - z) - y
    z_dot = x * y - beta * z
    u_dot = np.array([x_dot, y_dot, z_dot])
    return u_dot

class LorenzStepperRK4:
    def __init__(self, dt=0.01, *, sigma=10, rho=28, beta=8/3):
        self.dt = dt
        self.sigma = sigma
        self.rho = rho
        self.beta = beta
    
    def __call__(self, u_prev):
        lorenz_rhs_fixed = lambda u: lorenz_rhs(
            u,
            sigma=self.sigma,
            rho=self.rho,
            beta=self.beta,
        )
        k_1 = lorenz_rhs_fixed(u_prev)
        k_2 = lorenz_rhs_fixed(u_prev + 0.5 * self.dt * k_1)
        k_3 = lorenz_rhs_fixed(u_prev + 0.5 * self.dt * k_2)
        k_4 = lorenz_rhs_fixed(u_prev + self.dt * k_3)
        u_next = u_prev + self.dt * (k_1 + 2*k_2 + 2*k_3 + k_4)/6
        return u_next

def produce_trj(init, n_steps=5000):
    trj = [init,]
    u_current = init 
    for i in range(n_steps):
        u_current = lorenz_stepper(u_current)
        trj.append(u_current)
    trj = np.array(trj)
    return trj

def plot_trj(trj,save=False):
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(projection='3d')
    
    t = np.linspace(0, 1, len(trj))
    for i in range(len(trj) - 1):
        ax.plot(trj[i:i+2, 0], trj[i:i+2, 1], trj[i:i+2, 2],
                color=plt.cm.viridis(t[i]), lw=0.4, alpha=0.8)
    
    ax.set(xlabel='x', ylabel='y', zlabel='z')
    if save==True:
        plt.savefig("lorenz.pdf",bbox_inches="tight")
    plt.show()

def create_init_states(u_0,perturbation_fn,scale,N=50):
    u_0_s=[u_0]
    for i in range(N):
        u_0_s.append(perturbation_fn(u_0,scale))
    return u_0_s

def run_multiple_traj(u_0_s):
    trjs=[]
    for u_0 in u_0_s:
        trjs.append(produce_trj(u_0, 8000))
    return np.array(trjs)

def perturb_x_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3)*np.array([1,0,0]))

def perturb_x_gaussian(u_0,scale):
    return u_0*(1+scale*np.random.normal(0,scale,3)*np.array([1,0,0]))

def perturb_y_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3)*np.array([0,1,0]))

def perturb_z_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3)*np.array([0,0,1]))

def perturb_xy_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3)*np.array([1,1,0]))

def perturb_yz_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3)*np.array([0,1,1]))

def perturb_xyz_uniform(u_0,scale):
    return u_0*(1+scale*np.random.uniform(-1/2,1/2,3))

def find_linear_part(x, y):
    """Find the linear part of a curve using the knee/elbow point detection method.

    Normalises both axes to [0, 1] and identifies the knee point as the location
    of maximum deviation from the diagonal (y = x line). All points up to and
    including the knee are considered the linear region.

    Args:
        x: 1D array-like of x values.
        y: 1D array-like of y values, assumed to follow a curve with an initial
           linear region that transitions to a flat curve.

    Returns:
        linear_indices: Boolean array of the same length as x, where True indicates
                        the point that belong to the linear part of the curve.
    """
    # easier with normalised data
    x_norm = (x - x.min()) / (x.max() - x.min())
    y_norm = (y - y.min()) / (y.max() - y.min())

    # distance to the y=x line
    distances = y_norm - x_norm

    # point we want is where this distance is the biggest
    knee_idx = np.argmax(distances)

    # mask
    linear_indices = np.arange(len(x)) <= knee_idx

    return linear_indices

def lyapunov_pairwise(traj, dt, plot=True,save=False):
    sq_diff = (traj[:, None, :] - traj[None, :, :]) ** 2
    rmse = np.sqrt(sq_diff)
    logdist = np.log(np.maximum(rmse[np.triu_indices(rmse.shape[0], k=1)], 1e-11))
    mean_logdist = np.mean(logdist, axis=0)

    # extract times for better plotting
    time_index = np.arange(len(traj[0])) * dt
    
    # compute the lyapunov exponent
    linear_indices = find_linear_part(time_index, mean_logdist)
    slope, intercept = np.polyfit(
        time_index[linear_indices], mean_logdist[linear_indices], 1
    )
    lyapunov_exponent = np.round(slope, 3)

    # conditionally plot and print
    if plot:
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.set_ylabel("$log(RMSE)$",fontsize="x-large")
        ax.set_xlabel("Day",fontsize="x-large")

        for n in range(logdist.shape[0]):
            label = "Pairwise Differences" if n == 0 else None
            ax.plot(
                time_index, 
                logdist[n, :], 
                color="grey", 
                alpha=0.5,
                linewidth=1,
                zorder=1,
                label=label
            )

        ax.plot(
            time_index,
            mean_logdist,
            color="black",
            linewidth=2,
            zorder=2,
            label="Mean $log(RMSE)$",
        )

        fit_y = slope * time_index[linear_indices] + intercept
        ax.plot(
            time_index[linear_indices],
            fit_y,
            color="blue",
            linewidth=2,
            ls="--",
            zorder=3,
            label=f"Linear Part (λ = {lyapunov_exponent})",
        )

        ax.set_title("Log distances (Pairwise)",fontsize="x-large")
        ax.legend(loc="lower right",fontsize="x-large")
        plt.tight_layout()
        if save==True:
            plt.savefig("lyapunov_L63.pdf",bbox_inches="tight")
        plt.show()
        print(f"Estimated Lyapunov Exponent (λ): {lyapunov_exponent} dt^-1")

    return lyapunov_exponent

In [ ]:
def growth_rate_pairwise_lorenz(traj, dt, plot=True, save=False):
    """Estimate the upper-bound predictability limit and growth rate (alpha) for the Lorenz system.

    Computes the root-mean-square error (RMSE) for all unique pairs of trajectories
    and fits the Lorenz logistic growth model (dE/dt = alpha * E * (1 - E/E_inf)) 
    to estimate the error growth rate (alpha) and the saturation error (E_inf).

    Args:
        traj: 3D numpy array of trajectories with shape (N_trajectories, N_steps, 3).
        dt: Time step size used in the simulation.
        plot: Boolean to display the matplotlib figure.
        save: Boolean to save the plot as a PNG.

    Returns:
        alpha, E_inf: Estimated growth rate and saturation error, rounded to 3 decimal places.
    """
    sq_diff = (traj[:, None, :, :] - traj[None, :, :, :]) ** 2

    # mean over the 3 state variables (x, y, z)
    mean_sq_diff = np.mean(sq_diff, axis=-1)
    rmse = np.sqrt(mean_sq_diff)

    # pairwise extraction
    pairwise_rmse = rmse[np.triu_indices(rmse.shape[0], k=1)]
    logdist = np.log(np.maximum(pairwise_rmse, 1e-11)) # 1e-11 to avoid log(0)

    # We need mean_rmse for the curve fit, and mean_logdist for plotting
    mean_rmse = np.mean(pairwise_rmse, axis=0)
    mean_logdist = np.mean(logdist, axis=0)

    # Time index for the x-axis
    time_index = np.arange(len(traj[0])) * dt

    # Fit the solution of the logistic equation directly to the RMSE
    E0_fixed = mean_rmse[0]
    
    def lorenz_logistic_solution(t, alpha_param, E_inf_param):
        # Clip exponent to prevent overflow warnings during optimization
        exponent = np.clip(-alpha_param * t, -500, 500)
        return E_inf_param / (1.0 + ((E_inf_param - E0_fixed) / E0_fixed) * np.exp(exponent))

    max_E = np.max(mean_rmse)
    
    # Fit the curve against time instead of against E
    popt, _ = curve_fit(
        lorenz_logistic_solution, 
        time_index, 
        mean_rmse, 
        p0=[0.9, max_E], 
        bounds=([0.0, max_E * 0.1], [3.0, max_E * 4])
    )
    
    alpha = np.round(popt[0], 3)
    E_inf = np.round(popt[1], 3)
    
    # Conditionally plot and print
    if plot:
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.set_ylabel("$log(RMSE)$",fontsize="x-large")
        ax.set_xlabel("Day",fontsize="x-large")

        # Plot all pairwise log distances
        for n in range(logdist.shape[0]):
            label = "Pairwise Differences" if n == 0 else None
            ax.plot(
                time_index, 
                logdist[n, :], 
                color="grey", 
                alpha=0.5,
                linewidth=1,
                zorder=1,
                label=label
            )
            
        # Plot mean log distance
        ax.plot(
            time_index,
            mean_logdist,
            color="black",
            linewidth=2,
            zorder=2,
            label="Mean $log(RMSE)$",
        )

        # Calculate theoretical fit for plotting
        E0 = mean_rmse[0]
        E_theoretical = E_inf / (1 + ((E_inf - E0) / E0) * np.exp(-alpha * time_index))
        log_E_theoretical = np.log(np.maximum(E_theoretical, 1e-11))

        # Plot theoretical fit
        ax.plot(
            time_index,
            log_E_theoretical,
            color="blue",
            linewidth=2,
            ls="--",
            zorder=3,
            label=f"Fit (α = {alpha}, E_inf = {E_inf})",
        )

        ax.set_title("Lorenz-63 Mean Predictability (Pairwise Logistic Fit)",fontsize="x-large")
        ax.legend(loc="lower right",fontsize="x-large")
        plt.tight_layout()
        
        if save:
            plt.savefig("lorenz_logistic_fit.png", dpi=300)
            
        plt.show()
        
        print(f"Estimated Growth Rate (α): {alpha} dt^-1 | Saturation Error (E_inf): {E_inf}")
    
    return alpha, E_inf

In [ ]:
lorenz_stepper = LorenzStepperRK4()

## Smol tests

In [ ]:
u_0 = np.ones(3)
trj_test = produce_trj(u_0, 8000)
trj_test2 = produce_trj(trj_test[-1],8000)
plot_trj(trj_test,save=True)
plot_trj(trj_test2)

## Computing of lyapunov exponents

In [ ]:
u_0 = produce_trj(np.ones(3), np.random.randint(7000,8000))[-1]  #basically this is taking the a starting point that is IN the attractor, it's the same
# as when we take a real day from ERA5, if we don't do that we'll just observe the two trajectories going towards the attractor
#which isn't what we're interested in.

## With 50 traj

Real value is around 0.9

### Uniform perturbations

1e-3 on x

In [ ]:
u_0_s=create_init_states(u_0,perturb_x_uniform,1e-3,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt)
lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

1e-5 on x

In [ ]:
u_0_s=create_init_states(produce_trj(np.ones(3), np.random.randint(5000,8000))[-1],perturb_x_gaussian,1e-5,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt,save=True)
#lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
#lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
#growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

1e-7 on x

In [ ]:
u_0_s=create_init_states(u_0,perturb_x_uniform,1e-7,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt)
lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

1e-7 on z

In [ ]:
u_0_s=create_init_states(u_0,perturb_z_uniform,1e-7,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt)
lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

1e-7 on xy

In [ ]:
u_0_s=create_init_states(u_0,perturb_xy_uniform,1e-7,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt)
lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

1e-7 on xyz

In [ ]:
u_0_s=create_init_states(u_0,perturb_xyz_uniform,1e-7,N=50)
trajs=run_multiple_traj(u_0_s)
trajs_x=trajs[:,:,0]
trajs_y=trajs[:,:,1]
trajs_z=trajs[:,:,2]
lyap_x = lyapunov_pairwise(trajs_x, lorenz_stepper.dt)
lyap_y = lyapunov_pairwise(trajs_y, lorenz_stepper.dt)
lyap_z = lyapunov_pairwise(trajs_z, lorenz_stepper.dt)
growth_rate, E_inf=growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt)

In [ ]:
lyap_x=[]
lyap_y=[]
lyap_z=[]
for i in range(100):
    u_0 = produce_trj(np.ones(3), np.random.randint(5000,10000))[-1]
    u_0_s=create_init_states(u_0,perturb_z_uniform,1e-7,N=50)
    trajs=run_multiple_traj(u_0_s)
    trajs_x=trajs[:,:,0]
    trajs_y=trajs[:,:,1]
    trajs_z=trajs[:,:,2]
    lyap_x.append(lyapunov_pairwise(trajs_x, lorenz_stepper.dt,plot=False))
    lyap_y.append(lyapunov_pairwise(trajs_y, lorenz_stepper.dt,plot=False))
    lyap_z.append(lyapunov_pairwise(trajs_z, lorenz_stepper.dt,plot=False))

In [ ]:
growth_rates=[]
for i in range(500):
    u_0 = produce_trj(np.ones(3), np.random.randint(5000,10000))[-1]
    u_0_s=create_init_states(u_0,perturb_z_uniform,1e-7,N=50)
    trajs=run_multiple_traj(u_0_s)
    growth_rates.append(growth_rate_pairwise_lorenz(trajs,lorenz_stepper.dt,plot=False)[0])

In [ ]:
lyap_z=np.array(lyap_z)
lyap_y=np.array(lyap_y)
lyap_x=np.array(lyap_x)
lyap = np.concatenate([lyap_x,lyap_y,lyap_z])

In [ ]:
mean_val = np.mean(lyap)
median_val = np.median(lyap)

plt.hist(lyap, bins=30,color="black")

plt.axvline(mean_val, color="red",label=f'Mean: {mean_val:.3f}')
plt.axvline(median_val, color="green",label=f'Median: {median_val:.3f}')

plt.title('Histogram of Lyapunov Exponents')
plt.xlabel('Lyapunov Exponent')
plt.ylabel('Frequency')
plt.legend()

plt.show()

In [ ]:
mean_val = np.mean(growth_rates)
median_val = np.median(growth_rates)

plt.hist(growth_rates, bins=30,color="black")

plt.axvline(mean_val, color="red",label=f'Mean: {mean_val:.3f}')
plt.axvline(median_val, color="green",label=f'Median: {median_val:.3f}')

plt.title('Histogram of Growth rates')
plt.xlabel('Lyapunov Exponent')
plt.ylabel('Frequency')
plt.legend()

plt.show()

Conclusion : Our algorithm seems efficient to compute the lyapunov exponent, giving plausible values. The way we perturb the system does not seem to change much on the results (uniform perturbations seem to give better results tho), nor does the variable we compute the exponent on, but the Lorenz63 model is a very simple model so there is nothing to say that it's true for more complex models. We know atleast that our algorithm is somewhat robust to compute the exponent.